# Reclamações 03 · Camada Gold — Rankings

## 1. Objetivo e método

A Silver entregou as reclamações no grão distribuidora × tipologia × nível × mês, já limpas, com os meses anômalos tratados e a exclusão da CELESC registrada. Esta etapa fecha o universo, calcula os indicadores em janela móvel de 12 meses e produz os rankings que respondem às perguntas de negócio. A análise e a discussão dos resultados ficam no `04_analysis`.

### Indicadores

| Indicador | Fórmula | Papel |
|---|---|---|
| Procedentes por mil UCs | procedentes em 12 meses ÷ média de `NumCon` nos 12 meses × 1.000 | Ranking de variação |
| Recebidas por mil UCs | recebidas em 12 meses ÷ média de `NumCon` nos 12 meses × 1.000 | Ranking de variação, em paralelo |
| Taxa de procedência | procedentes ÷ (procedentes + improcedentes) | Diagnóstico, sem ranking |

O ranking de recebidas existe porque a procedência é classificada pela própria distribuidora. Uma empresa pode reduzir as procedentes classificando mais reclamações como improcedentes, e as recebidas mostram se o cliente de fato reclama menos. A taxa usa procedentes mais improcedentes no denominador, e não as recebidas, porque parte das recebidas ainda não tem conclusão.

O denominador é o `NumCon` da base de continuidade, o mesmo que define o porte da distribuidora e que pesa o DEC e o FEC.

### Recortes

| Recorte | Conteúdo | Por que existe |
|---|---|---|
| `comercial_estrito` | Todo o comercial estrito | Resposta direta à P2: o volume de problemas comerciais que o cliente enfrenta |
| `faturamento` | Grupo `10204` | Cerca de metade do comercial estrito; afeta todas as classes |
| `pagamento` | Grupo `10206` | Afeta todas as classes |
| `outras_comerciais` | Restante do comercial estrito, inclusive Conexão e Geração Distribuída | Evita que grupos de volume baixo por distribuidora gerem rankings instáveis |
| `qualidade` | Grupo `10209` | Cerca de 94% das procedentes da família 102; confronto com a continuidade na P5 |

Os recortes vêm do atributo `grupo_ranking` da `dim_tipologia`; a Gold apenas filtra.

### Janelas e ranking

- Janelas móveis de 12 meses encerradas em dezembro de 2024, junho de 2025, dezembro de 2025 e junho de 2026. A janela de junho de 2024 não existe, porque a série começa em janeiro de 2024.
- Uma janela só é válida com os 12 meses de `NumCon` e de envio de reclamações.
- O ranking compara a janela de dezembro de 2024 com a de junho de 2026. A posição é dada pela variação percentual, da maior redução para o maior aumento; empate se resolve pela variação absoluta.
- Entram no ranking as distribuidoras de grande porte, menos as registradas em `exclusao_ranking_manifestacao`.
- Somente o nível 1 entra: a ouvidoria recebe reclamações que já passaram pelo nível 1, e somar os dois níveis contaria duas vezes o mesmo problema.
- Sensibilidade: o mesmo ranking é calculado com os valores publicados, sem a imputação da Silver.

### Continuidade na mesma janela

A P5 confronta o ranking de Qualidade com o de continuidade. A Gold de continuidade compara anos civis (2022 e 2025) e não tem 2026, então o DEC-FI e o FEC-FI são recalculados aqui nas mesmas janelas, a partir da `silver.fato_continuidade_mensal`, com as equações 41 a 46 do PRODIST Módulo 8. Os notebooks de continuidade não mudam.

### Tabelas produzidas

| Tabela | Grão | Conteúdo |
|---|---|---|
| `fato_reclamacao_janela` | distribuidora × recorte × fim de janela | Quantidades em 12 meses, UCs médias, indicadores, taxa de procedência, validade da janela e flag de exclusão; no recorte `comercial_estrito`, também as quantidades do nível 2 e a taxa de escalada |
| `ranking_reclamacoes` | recorte × medida × distribuidora | Indicador inicial e final, variações, posição, volume e sensibilidade sem imputação |
| `continuidade_janela` | distribuidora × fim de janela | DEC-FI e FEC-FI em 12 meses pelas equações normativas |
| `ranking_continuidade` | indicador × distribuidora | Variação e posição do DEC-FI e do FEC-FI entre as mesmas janelas |

### Ordem de execução

Este notebook roda depois de `complaints/02_silver_complaints` e de `continuity/01_silver_continuity`, e antes de `complaints/04_analysis`.

## 2. Configuração

In [ ]:
import os
import sys

from pyspark.sql import functions as F
from pyspark.sql import Window

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_GOLD, SCHEMA_SILVER

SILVER = f"{CATALOG}.{SCHEMA_SILVER}"
GOLD = f"{CATALOG}.{SCHEMA_GOLD}"

# Only the call centre level enters the rankings
NIVEL_RANKING = 1

# Scopes of the rankings: the strict commercial total plus the groups of dim_tipologia
RECORTE_TOTAL = "comercial_estrito"
GRUPOS_RANKING = ["faturamento", "pagamento", "outras_comerciais", "qualidade"]
RECORTES = [RECORTE_TOTAL] + GRUPOS_RANKING

# Measures that get a ranking; the complaint rate is a diagnostic only
MEDIDAS_RANKING = ["procedentes", "recebidas"]
QUANTIDADES = ["recebidas", "procedentes", "improcedentes"]

MESES_JANELA = 12
CASAS_INDICADOR = 4
CASAS_CONTINUIDADE = 2   # PRODIST Module 8 states two decimal places at each step

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_GOLD}")


def indice_mes(coluna):
    """Month count since year zero, so that month distance is a plain subtraction."""
    return (F.floor(F.col(coluna) / 100) * 12 + F.col(coluna) % 100).cast("int")


tempo = spark.table(f"{SILVER}.dim_tempo")
primeiro_mes = tempo.agg(F.min("ano_mes")).first()[0]

# Window ends with twelve months inside the series; June 2024 has only six
FINS_JANELA = sorted(
    r["ano_mes"] for r in tempo.filter("ind_fim_janela")
    .withColumn("idx", indice_mes("ano_mes"))
    .filter(F.col("idx") - (MESES_JANELA - 1) >= F.lit((primeiro_mes // 100) * 12 + primeiro_mes % 100))
    .select("ano_mes").collect())
FIM_INICIAL, FIM_FINAL = FINS_JANELA[0], FINS_JANELA[-1]

print(f"Origem..........: {SILVER}")
print(f"Destino.........: {GOLD}")
print(f"Recortes........: {RECORTES}")
print(f"Fins de janela..: {FINS_JANELA}")
print(f"Ranking.........: {FIM_INICIAL} contra {FIM_FINAL}")

## 3. Universo e denominador

O universo são as distribuidoras de grande porte da `dim_distribuidora`. As registradas em `exclusao_ranking_manifestacao` continuam nas tabelas de janela, com a flag `excluida_ranking`, para que a exclusão fique visível, mas não entram nos rankings.

O denominador mensal é a soma do `NumCon` dos conjuntos da distribuidora no mês.

In [ ]:
dim_dx = spark.table(f"{SILVER}.dim_distribuidora")
exclusao = spark.table(f"{SILVER}.exclusao_ranking_manifestacao").select("num_cnpj", "motivo")

universo = (dim_dx
    .filter(F.col("grande_porte"))
    .select("num_cnpj", "sig_agente")
    .join(exclusao, "num_cnpj", "left")
    .withColumn("excluida_ranking", F.col("motivo").isNotNull())
    .drop("motivo"))

meses_serie = tempo.select("ano_mes")

ucs_mensal = (spark.table(f"{SILVER}.fato_continuidade_mensal")
    .withColumn("ano_mes", F.col("ano") * 100 + F.col("mes"))
    .join(meses_serie, "ano_mes", "inner")
    .groupBy("num_cnpj", "ano_mes")
    .agg(F.sum("num_con").alias("ucs")))

print(f"distribuidoras no universo..: {universo.count()}")
print(f"excluidas do ranking........: {universo.filter('excluida_ranking').count()}")

display(universo
        .join(ucs_mensal, "num_cnpj", "left")
        .groupBy("sig_agente", "excluida_ranking")
        .agg(F.countDistinct("ano_mes").alias("meses_com_ucs"),
             F.round(F.avg("ucs"), 0).cast("long").alias("ucs_media"))
        .orderBy(F.col("ucs_media").desc()))

## 4. Série mensal por recorte

A série é montada em grade completa (distribuidora × recorte × mês), com zero onde não houve reclamação no recorte. Isso é diferente de ausência de envio: um mês sem nenhuma linha publicada da distribuidora no nível 1 é marcado como sem envio e invalida as janelas que o contêm.

As quantidades usadas (`qtd_*`) incluem a imputação da Silver; as publicadas (`qtd_*_publicada`) seguem ao lado para a sensibilidade.

O nível 2 entra apenas no recorte `comercial_estrito`, como referência para a P3, e não participa de nenhum ranking. Por grupo ou por tipologia a comparação entre níveis não é confiável: em algumas distribuidoras o nível 2 supera o nível 1 inteiro em tipologias específicas, todos os meses, sinal de reporte inconsistente entre os níveis. No total do comercial estrito o nível 2 é menor que o nível 1 em todas as distribuidoras, e a validação confere isso.

In [ ]:
fato = spark.table(f"{SILVER}.fato_manifestacao")
tipologia = spark.table(f"{SILVER}.dim_tipologia").select(
    "cod_tipologia", "ind_comercial_estrito", "grupo_ranking")

nivel_1 = fato.filter(F.col("nivel") == NIVEL_RANKING).join(F.broadcast(tipologia), "cod_tipologia")

COLUNAS_QTD = ([f"qtd_{q}" for q in QUANTIDADES] +
               [f"qtd_{q}_publicada" for q in QUANTIDADES])

total = nivel_1.filter("ind_comercial_estrito").withColumn("recorte", F.lit(RECORTE_TOTAL))
grupos = (nivel_1.filter(F.col("grupo_ranking").isin(GRUPOS_RANKING))
          .withColumn("recorte", F.col("grupo_ranking")))

por_recorte = (total.unionByName(grupos)
    .groupBy("num_cnpj", "recorte", "ano_mes")
    .agg(*[F.sum(c).alias(c) for c in COLUNAS_QTD]))

# Level 2 is kept only for the strict commercial total, as a reference for P3
COLUNAS_N2 = [f"qtd_{q}_nivel_2" for q in QUANTIDADES]
nivel_2_total = (fato.filter(F.col("nivel") == 2)
    .join(F.broadcast(tipologia), "cod_tipologia")
    .filter("ind_comercial_estrito")
    .groupBy("num_cnpj", "ano_mes")
    .agg(*[F.sum(f"qtd_{q}").alias(f"qtd_{q}_nivel_2") for q in QUANTIDADES])
    .withColumn("recorte", F.lit(RECORTE_TOTAL)))

# A month is sent when the company published at least one level 1 complaint
envio = (fato.filter(F.col("nivel") == NIVEL_RANKING)
    .groupBy("num_cnpj", "ano_mes")
    .agg((F.sum("qtd_recebidas_publicada") > 0).alias("com_envio")))

recortes_df = spark.createDataFrame([(r,) for r in RECORTES], "recorte string")

mensal = (universo.select("num_cnpj")
    .crossJoin(recortes_df)
    .crossJoin(meses_serie)
    .join(por_recorte, ["num_cnpj", "recorte", "ano_mes"], "left")
    .na.fill(0, subset=COLUNAS_QTD)
    .join(nivel_2_total, ["num_cnpj", "recorte", "ano_mes"], "left")
    .withColumn("_total", F.col("recorte") == RECORTE_TOTAL)
    .select("*", *[F.when(F.col("_total"), F.coalesce(F.col(c), F.lit(0))).alias(f"_{c}")
                   for c in COLUNAS_N2])
    .drop(*COLUNAS_N2, "_total")
    .withColumnsRenamed({f"_{c}": c for c in COLUNAS_N2})
    .join(ucs_mensal, ["num_cnpj", "ano_mes"], "left")
    .join(envio, ["num_cnpj", "ano_mes"], "left")
    .withColumn("com_envio", F.coalesce(F.col("com_envio"), F.lit(False))))

sem_envio = (mensal.filter(F.col("recorte") == RECORTE_TOTAL)
    .filter(~F.col("com_envio") | F.col("ucs").isNull())
    .join(universo, "num_cnpj")
    .select("sig_agente", "ano_mes", "com_envio", "ucs")
    .orderBy("sig_agente", "ano_mes"))

print(f"meses sem envio ou sem UCs no universo: {sem_envio.count()}")
display(sem_envio)

## 5. `fato_reclamacao_janela`

Cada janela soma os 12 meses que terminam no seu fim. O denominador é a média mensal das UCs na janela, a mesma lógica da agregação anual do PRODIST: com universo estável, o resultado coincide com dividir pela UC de um mês; quando o universo varia, a média evita que um mês isolado defina o denominador.

In [ ]:
fins_df = spark.createDataFrame([(f,) for f in FINS_JANELA], "fim_janela int")

janela_meses = (fins_df
    .crossJoin(meses_serie)
    .filter((indice_mes("ano_mes") <= indice_mes("fim_janela")) &
            (indice_mes("ano_mes") > indice_mes("fim_janela") - MESES_JANELA)))

agregado = (mensal
    .join(janela_meses, "ano_mes")
    .groupBy("num_cnpj", "recorte", "fim_janela")
    .agg(*[F.sum(c).alias(c) for c in COLUNAS_QTD + COLUNAS_N2],
         F.countDistinct("ano_mes").alias("meses"),
         F.count("ucs").alias("meses_com_ucs"),
         F.sum(F.col("com_envio").cast("int")).alias("meses_com_envio"),
         F.avg("ucs").alias("ucs_media")))


def por_mil(coluna):
    return F.round(F.col(coluna) / F.col("ucs_media") * 1000, CASAS_INDICADOR)


def taxa(proc, improc):
    base = F.col(proc) + F.col(improc)
    return F.when(base > 0, F.round(F.col(proc) / base, CASAS_INDICADOR))


fato_janela = (agregado
    .withColumn("janela_valida",
                (F.col("meses") == MESES_JANELA) &
                (F.col("meses_com_ucs") == MESES_JANELA) &
                (F.col("meses_com_envio") == MESES_JANELA))
    .withColumn("ucs_media", F.round("ucs_media", 0).cast("long"))
    .withColumn("recebidas_por_mil", por_mil("qtd_recebidas"))
    .withColumn("procedentes_por_mil", por_mil("qtd_procedentes"))
    .withColumn("taxa_procedencia", taxa("qtd_procedentes", "qtd_improcedentes"))
    .withColumn("recebidas_por_mil_publicada", por_mil("qtd_recebidas_publicada"))
    .withColumn("procedentes_por_mil_publicada", por_mil("qtd_procedentes_publicada"))
    .withColumn("taxa_procedencia_publicada",
                taxa("qtd_procedentes_publicada", "qtd_improcedentes_publicada"))
    .withColumn("recebidas_nivel_2_por_mil", por_mil("qtd_recebidas_nivel_2"))
    .withColumn("taxa_escalada",
                F.when(F.col("qtd_recebidas_publicada") > 0,
                       F.round(F.col("qtd_recebidas_nivel_2") / F.col("qtd_recebidas_publicada"),
                               CASAS_INDICADOR)))
    .join(universo.select("num_cnpj", "excluida_ranking"), "num_cnpj")
    .select("num_cnpj", "recorte", "fim_janela", "meses_com_ucs", "meses_com_envio",
            "janela_valida", "excluida_ranking", "ucs_media", *COLUNAS_QTD,
            "recebidas_por_mil", "procedentes_por_mil", "taxa_procedencia",
            "recebidas_por_mil_publicada", "procedentes_por_mil_publicada",
            "taxa_procedencia_publicada", *COLUNAS_N2,
            "recebidas_nivel_2_por_mil", "taxa_escalada"))

(fato_janela.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.fato_reclamacao_janela"))

janela_lida = spark.table(f"{GOLD}.fato_reclamacao_janela")
print(f"fato_reclamacao_janela: {janela_lida.count():,} linhas")

display(janela_lida
        .filter(F.col("recorte") == RECORTE_TOTAL)
        .groupBy("fim_janela")
        .agg(F.sum(F.col("janela_valida").cast("int")).alias("janelas_validas"),
             F.sum("qtd_procedentes").alias("procedentes"),
             F.sum("qtd_recebidas").alias("recebidas"),
             F.round(F.sum("qtd_procedentes") / F.sum("ucs_media") * 1000, 4).alias("procedentes_por_mil_universo"))
        .orderBy("fim_janela"))

## 6. `ranking_reclamacoes`

Dez rankings: cinco recortes, cada um por procedentes e por recebidas. A distribuidora entra quando não está excluída, tem as duas janelas válidas e indicador inicial positivo; sem valor inicial, a variação percentual não tem significado.

A posição 1 é a maior redução. O volume absoluto das duas janelas segue ao lado, porque nos recortes de volume baixo por distribuidora a variação percentual é mais sensível a oscilação. As colunas `_sem_imputacao` repetem o cálculo com os valores publicados, e `variacao_posicao` mostra quantas posições a imputação deslocou.

In [ ]:
elegivel = (janela_lida
    .filter(~F.col("excluida_ranking") & F.col("janela_valida"))
    .filter(F.col("fim_janela").isin(FIM_INICIAL, FIM_FINAL)))


def ranking_medida(medida):
    """Variation between the first and last window for one measure, with and without imputation."""
    valor = f"{medida}_por_mil"
    ini = (elegivel.filter(F.col("fim_janela") == FIM_INICIAL)
        .select("num_cnpj", "recorte",
                F.col(valor).alias("indicador_inicio"),
                F.col(f"{valor}_publicada").alias("indicador_inicio_sem_imputacao"),
                F.col(f"qtd_{medida}").alias("volume_inicio")))
    fim = (elegivel.filter(F.col("fim_janela") == FIM_FINAL)
        .select("num_cnpj", "recorte",
                F.col(valor).alias("indicador_fim"),
                F.col(f"{valor}_publicada").alias("indicador_fim_sem_imputacao"),
                F.col(f"qtd_{medida}").alias("volume_fim")))
    return (ini.join(fim, ["num_cnpj", "recorte"])
        .withColumn("medida", F.lit(medida)))


def variacao(fim, ini):
    return F.when(F.col(ini) > 0, F.round((F.col(fim) / F.col(ini) - 1) * 100, 2))


pares = ranking_medida(MEDIDAS_RANKING[0])
for medida in MEDIDAS_RANKING[1:]:
    pares = pares.unionByName(ranking_medida(medida))

pares = (pares
    .filter(F.col("indicador_inicio") > 0)
    .withColumn("var_abs", F.round(F.col("indicador_fim") - F.col("indicador_inicio"), CASAS_INDICADOR))
    .withColumn("var_pct", variacao("indicador_fim", "indicador_inicio"))
    .withColumn("var_abs_sem_imputacao",
                F.round(F.col("indicador_fim_sem_imputacao") - F.col("indicador_inicio_sem_imputacao"),
                        CASAS_INDICADOR))
    .withColumn("var_pct_sem_imputacao",
                variacao("indicador_fim_sem_imputacao", "indicador_inicio_sem_imputacao")))

por_medida = Window.partitionBy("recorte", "medida")
ranking = (pares
    .withColumn("posicao", F.rank().over(por_medida.orderBy("var_pct", "var_abs")))
    .withColumn("posicao_sem_imputacao",
                F.rank().over(por_medida.orderBy("var_pct_sem_imputacao", "var_abs_sem_imputacao")))
    .withColumn("variacao_posicao", F.col("posicao_sem_imputacao") - F.col("posicao"))
    .select("recorte", "medida", "num_cnpj", "posicao",
            "indicador_inicio", "indicador_fim", "var_abs", "var_pct",
            "volume_inicio", "volume_fim",
            "indicador_inicio_sem_imputacao", "indicador_fim_sem_imputacao",
            "var_abs_sem_imputacao", "var_pct_sem_imputacao",
            "posicao_sem_imputacao", "variacao_posicao"))

(ranking.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.ranking_reclamacoes"))

ranking_lido = spark.table(f"{GOLD}.ranking_reclamacoes")
nomes = dim_dx.select("num_cnpj", "sig_agente")

print(f"ranking_reclamacoes: {ranking_lido.count():,} linhas")
display(ranking_lido.groupBy("recorte", "medida").count().orderBy("recorte", "medida"))

display(ranking_lido
        .filter((F.col("recorte") == RECORTE_TOTAL) & (F.col("medida") == "procedentes"))
        .join(nomes, "num_cnpj")
        .select("posicao", "sig_agente", "indicador_inicio", "indicador_fim",
                "var_pct", "var_abs", "volume_inicio", "volume_fim",
                "posicao_sem_imputacao")
        .orderBy("posicao"))

## 7. Continuidade na mesma janela

O DEC-FI e o FEC-FI são agregados em dois passos, como na Gold de continuidade:

| Passo | Equações | Fórmula |
|---|---|---|
| Global mensal | 41 a 43 | `DECGn = Σ(DECi × NUCi) / NUCGn`, com `NUCGn = Σ NUCi` |
| Janela de 12 meses | 44 a 46 | `DECGk = Σ(DECGn × NUCGn) / NUCGk`, com `NUCGk = (Σ NUCGn) / k` |

A norma define o segundo passo para o ano civil; aqui ele é aplicado à janela de 12 meses. Para a janela encerrada em dezembro de 2025, que coincide com o ano civil, o resultado tem de reproduzir a `fato_continuidade_anual` da Gold de continuidade, e a validação confere isso.

O ranking de continuidade usa o mesmo conjunto de distribuidoras do ranking de Qualidade, para que as posições sejam comparáveis na P5.

In [ ]:
INDICADORES_CONT = ["dec_fi", "fec_fi"]

cont = (spark.table(f"{SILVER}.fato_continuidade_mensal")
    .withColumn("ano_mes", F.col("ano") * 100 + F.col("mes"))
    .join(meses_serie, "ano_mes", "inner")
    .join(universo.select("num_cnpj"), "num_cnpj"))

# Equations 41 to 43: weighted mean of the sets, weighted by that month's NUC
global_mensal = (cont
    .groupBy("num_cnpj", "ano_mes")
    .agg(F.sum("num_con").alias("nuc_g"),
         *[F.sum(F.col(n) * F.col("num_con")).alias(f"_s_{n}") for n in INDICADORES_CONT]))
for n in INDICADORES_CONT:
    global_mensal = global_mensal.withColumn(
        n, F.round(F.col(f"_s_{n}") / F.col("nuc_g"), CASAS_CONTINUIDADE))

# Equations 44 to 46 over the 12-month window: the denominator is the average universe
cont_janela = (global_mensal
    .join(janela_meses, "ano_mes")
    .groupBy("num_cnpj", "fim_janela")
    .agg(F.count("*").alias("meses"),
         F.sum("nuc_g").alias("_soma_nuc"),
         *[F.sum(F.col(n) * F.col("nuc_g")).alias(f"_p_{n}") for n in INDICADORES_CONT])
    .withColumn("nuc_media", F.round(F.col("_soma_nuc") / F.col("meses"), 0)))
for n in INDICADORES_CONT:
    cont_janela = cont_janela.withColumn(
        n, F.round(F.col(f"_p_{n}") / F.col("nuc_media"), CASAS_CONTINUIDADE))

cont_janela = (cont_janela
    .withColumn("janela_valida", F.col("meses") == MESES_JANELA)
    .select("num_cnpj", "fim_janela", "meses", "janela_valida",
            F.col("nuc_media").cast("long").alias("nuc_media"), *INDICADORES_CONT))

(cont_janela.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.continuidade_janela"))

cont_lida = spark.table(f"{GOLD}.continuidade_janela")
print(f"continuidade_janela: {cont_lida.count():,} linhas")

In [ ]:
# Same set of companies as the Quality complaints ranking, so that positions compare
conjunto_qualidade = (ranking_lido
    .filter((F.col("recorte") == "qualidade") & (F.col("medida") == "procedentes"))
    .select("num_cnpj"))

base_cont = cont_lida.filter(F.col("janela_valida")).join(conjunto_qualidade, "num_cnpj")

partes = []
for n in INDICADORES_CONT:
    ini = base_cont.filter(F.col("fim_janela") == FIM_INICIAL).select(
        "num_cnpj", F.col(n).alias("valor_inicio"))
    fim = base_cont.filter(F.col("fim_janela") == FIM_FINAL).select(
        "num_cnpj", F.col(n).alias("valor_fim"))
    partes.append(ini.join(fim, "num_cnpj").withColumn("indicador", F.lit(n)))

ranking_cont = partes[0]
for parte in partes[1:]:
    ranking_cont = ranking_cont.unionByName(parte)

ranking_cont = (ranking_cont
    .withColumn("var_abs", F.round(F.col("valor_fim") - F.col("valor_inicio"), CASAS_CONTINUIDADE))
    .withColumn("var_pct", variacao("valor_fim", "valor_inicio"))
    .withColumn("posicao", F.rank().over(
        Window.partitionBy("indicador").orderBy("var_pct", "var_abs")))
    .select("indicador", "num_cnpj", "posicao", "valor_inicio", "valor_fim", "var_abs", "var_pct"))

(ranking_cont.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.ranking_continuidade"))

ranking_cont_lido = spark.table(f"{GOLD}.ranking_continuidade")
print(f"ranking_continuidade: {ranking_cont_lido.count():,} linhas")

display(ranking_cont_lido
        .filter(F.col("indicador") == "dec_fi")
        .join(nomes, "num_cnpj")
        .select("posicao", "sig_agente", "valor_inicio", "valor_fim", "var_pct")
        .orderBy("posicao"))

## 8. Catálogo

In [ ]:
COMENTARIOS = {
    f"{GOLD}.fato_reclamacao_janela": (
        "Reclamacoes do nivel 1 em janelas moveis de 12 meses por distribuidora de grande porte e "
        "recorte, com indicadores por mil UCs e taxa de procedencia. Mantem as distribuidoras "
        "excluidas do ranking, sinalizadas.",
        {
            "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres",
            "recorte": "comercial_estrito, faturamento, pagamento, outras_comerciais ou qualidade",
            "fim_janela": "Ultimo mes da janela de 12 meses, formato AAAAMM",
            "meses_com_ucs": "Meses da janela com NumCon informado",
            "meses_com_envio": "Meses da janela com reclamacao publicada pela distribuidora no nivel 1",
            "janela_valida": "Verdadeiro quando os 12 meses tem UCs e envio de reclamacoes",
            "excluida_ranking": "Verdadeiro quando a distribuidora consta de exclusao_ranking_manifestacao",
            "ucs_media": "Media mensal da soma de NumCon na janela; denominador dos indicadores",
            "qtd_recebidas": "Recebidas na janela, com a imputacao da Silver",
            "qtd_procedentes": "Procedentes na janela, com a imputacao da Silver",
            "qtd_improcedentes": "Improcedentes na janela, com a imputacao da Silver",
            "qtd_recebidas_publicada": "Recebidas na janela conforme publicadas",
            "qtd_procedentes_publicada": "Procedentes na janela conforme publicadas",
            "qtd_improcedentes_publicada": "Improcedentes na janela conforme publicadas",
            "recebidas_por_mil": "Recebidas por mil UCs na janela",
            "procedentes_por_mil": "Procedentes por mil UCs na janela",
            "taxa_procedencia": "Procedentes divididas por procedentes mais improcedentes, de 0 a 1",
            "recebidas_por_mil_publicada": "Recebidas por mil UCs com valores publicados, sem imputacao",
            "procedentes_por_mil_publicada": "Procedentes por mil UCs com valores publicados, sem imputacao",
            "taxa_procedencia_publicada": "Taxa de procedencia com valores publicados, sem imputacao",
            "qtd_recebidas_nivel_2": "Recebidas no nivel 2 (ouvidoria) na janela; so no recorte comercial_estrito",
            "qtd_procedentes_nivel_2": "Procedentes no nivel 2 na janela; so no recorte comercial_estrito",
            "qtd_improcedentes_nivel_2": "Improcedentes no nivel 2 na janela; so no recorte comercial_estrito",
            "recebidas_nivel_2_por_mil": "Recebidas no nivel 2 por mil UCs; so no recorte comercial_estrito",
            "taxa_escalada": "Recebidas no nivel 2 divididas pelas recebidas publicadas no nivel 1; so no recorte comercial_estrito",
        }),
    f"{GOLD}.ranking_reclamacoes": (
        "Rankings de variacao das reclamacoes por mil UCs entre a primeira e a ultima janela, por "
        "recorte e medida. Posicao 1 e a maior reducao; inclui a sensibilidade sem imputacao.",
        {
            "recorte": "comercial_estrito, faturamento, pagamento, outras_comerciais ou qualidade",
            "medida": "procedentes ou recebidas",
            "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres",
            "posicao": "Posicao pela variacao percentual, com a variacao absoluta como desempate",
            "indicador_inicio": "Indicador por mil UCs na primeira janela",
            "indicador_fim": "Indicador por mil UCs na ultima janela",
            "var_abs": "Indicador final menos inicial",
            "var_pct": "Variacao percentual entre o indicador inicial e o final",
            "volume_inicio": "Quantidade absoluta na primeira janela",
            "volume_fim": "Quantidade absoluta na ultima janela",
            "indicador_inicio_sem_imputacao": "Indicador inicial com valores publicados",
            "indicador_fim_sem_imputacao": "Indicador final com valores publicados",
            "var_abs_sem_imputacao": "Variacao absoluta com valores publicados",
            "var_pct_sem_imputacao": "Variacao percentual com valores publicados",
            "posicao_sem_imputacao": "Posicao calculada com valores publicados",
            "variacao_posicao": "Posicao sem imputacao menos posicao com imputacao",
        }),
    f"{GOLD}.continuidade_janela": (
        "DEC-FI e FEC-FI por distribuidora de grande porte em janelas moveis de 12 meses, pelas "
        "Equacoes 41 a 46 do PRODIST Modulo 8, alinhados as janelas das reclamacoes.",
        {
            "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres",
            "fim_janela": "Ultimo mes da janela de 12 meses, formato AAAAMM",
            "meses": "Meses com indicador na janela",
            "janela_valida": "Verdadeiro quando a janela tem 12 meses",
            "nuc_media": "Media mensal do universo de UCs na janela (Eq. 46)",
            "dec_fi": "DEC de falha interna na janela, em horas",
            "fec_fi": "FEC de falha interna na janela, em interrupcoes",
        }),
    f"{GOLD}.ranking_continuidade": (
        "Ranking de variacao do DEC-FI e do FEC-FI entre as mesmas janelas das reclamacoes, no "
        "mesmo conjunto de distribuidoras do ranking de Qualidade.",
        {
            "indicador": "dec_fi ou fec_fi",
            "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres",
            "posicao": "Posicao pela variacao percentual; 1 e a maior reducao",
            "valor_inicio": "Indicador na primeira janela",
            "valor_fim": "Indicador na ultima janela",
            "var_abs": "Valor final menos inicial",
            "var_pct": "Variacao percentual entre o valor inicial e o final",
        }),
}

for tabela, (descricao, colunas) in COMENTARIOS.items():
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{descricao}'")
    for coluna, texto in colunas.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {sum(len(c) for _, c in COMENTARIOS.values())} colunas em {len(COMENTARIOS)} tabelas")

## 9. Validação

Os testes conferem a chave, a completude do universo, a aritmética entre recortes, a conservação do volume entre Silver e Gold, a aplicação da exclusão, a coerência da sensibilidade e a reprodução da Gold de continuidade.

In [ ]:
testes = []
n_universo = universo.count()

linhas = janela_lida.count()
unicas = janela_lida.select("num_cnpj", "recorte", "fim_janela").distinct().count()
testes.append(("chave unica em fato_reclamacao_janela",
               linhas == unicas,
               f"{linhas:,} linhas para {unicas:,} chaves"))

esperadas = n_universo * len(RECORTES) * len(FINS_JANELA)
testes.append(("grade completa do universo",
               linhas == esperadas,
               f"{linhas} linhas, esperadas {esperadas}"))

# The three commercial groups must add up to the strict commercial total
soma_grupos = (janela_lida.filter(F.col("recorte").isin("faturamento", "pagamento", "outras_comerciais"))
    .groupBy("num_cnpj", "fim_janela")
    .agg(*[F.sum(c).alias(c) for c in COLUNAS_QTD]))
total_lido = (janela_lida.filter(F.col("recorte") == RECORTE_TOTAL)
    .select("num_cnpj", "fim_janela", *[F.col(c).alias(f"t_{c}") for c in COLUNAS_QTD]))
divergentes = (soma_grupos.join(total_lido, ["num_cnpj", "fim_janela"])
    .filter(" OR ".join(f"{c} <> t_{c}" for c in COLUNAS_QTD)).count())
testes.append(("grupos comerciais somam o total",
               divergentes == 0,
               f"{divergentes} distribuidora-janela divergentes"))

# Volume of the last window against the Silver fact
meses_fim = [r["ano_mes"] for r in janela_meses.filter(F.col("fim_janela") == FIM_FINAL).collect()]
esperado = (nivel_1.filter("ind_comercial_estrito")
    .filter(F.col("ano_mes").isin(meses_fim))
    .join(universo.select("num_cnpj"), "num_cnpj")
    .agg(F.sum("qtd_procedentes")).first()[0] or 0)
obtido = (janela_lida.filter((F.col("recorte") == RECORTE_TOTAL) & (F.col("fim_janela") == FIM_FINAL))
    .agg(F.sum("qtd_procedentes")).first()[0] or 0)
testes.append(("volume conservado entre Silver e Gold",
               esperado == obtido,
               f"procedentes {obtido:,}/{esperado:,} na janela {FIM_FINAL}"))

taxa_fora = janela_lida.filter((F.col("taxa_procedencia") < 0) | (F.col("taxa_procedencia") > 1)).count()
testes.append(("taxa de procedencia entre 0 e 1",
               taxa_fora == 0,
               f"{taxa_fora} linhas fora do intervalo"))

n2_fora = janela_lida.filter((F.col("recorte") != RECORTE_TOTAL) &
                             F.col("qtd_recebidas_nivel_2").isNotNull()).count()
n2_acima = janela_lida.filter((F.col("recorte") == RECORTE_TOTAL) &
                              (F.col("qtd_recebidas_nivel_2") > F.col("qtd_recebidas_publicada"))).count()
testes.append(("nivel 2 so no total e menor que o nivel 1",
               n2_fora == 0 and n2_acima == 0,
               f"{n2_fora} linhas de grupo com nivel 2; {n2_acima} janelas com N2 acima de N1"))

excluidas_no_ranking = (ranking_lido.join(exclusao, "num_cnpj").count())
testes.append(("exclusao aplicada ao ranking",
               excluidas_no_ranking == 0,
               f"{excluidas_no_ranking} linhas de distribuidoras excluidas"))

linhas_rk = ranking_lido.count()
unicas_rk = ranking_lido.select("recorte", "medida", "num_cnpj").distinct().count()
testes.append(("chave unica em ranking_reclamacoes",
               linhas_rk == unicas_rk,
               f"{linhas_rk} linhas para {unicas_rk} chaves"))

rankings = ranking_lido.select("recorte", "medida").distinct().count()
testes.append(("dez rankings produzidos",
               rankings == len(RECORTES) * len(MEDIDAS_RANKING),
               f"{rankings} combinacoes de recorte e medida"))

posicoes = (ranking_lido.groupBy("recorte", "medida")
    .agg(F.min("posicao").alias("minimo"), F.max("posicao").alias("maximo"), F.count("*").alias("n"))
    .filter((F.col("minimo") != 1) | (F.col("maximo") > F.col("n"))).count())
testes.append(("posicoes de 1 ao numero de distribuidoras",
               posicoes == 0,
               f"{posicoes} rankings com posicao fora do intervalo"))

# Without imputed months in the two compared windows, both versions must agree
meses_comparados = [r["ano_mes"] for r in janela_meses
                    .filter(F.col("fim_janela").isin(FIM_INICIAL, FIM_FINAL)).collect()]
com_imputacao = (fato.filter(F.col("imputado") & F.col("ano_mes").isin(meses_comparados))
    .select("num_cnpj").distinct())
incoerentes = (ranking_lido.join(com_imputacao, "num_cnpj", "left_anti")
    .filter((F.col("var_pct") != F.col("var_pct_sem_imputacao")) |
            (F.col("indicador_fim") != F.col("indicador_fim_sem_imputacao"))).count())
testes.append(("sensibilidade igual sem meses imputados",
               incoerentes == 0,
               f"{incoerentes} linhas divergentes sem imputacao"))

cont_invalidas = cont_lida.filter(~F.col("janela_valida")).count()
testes.append(("continuidade com 12 meses em toda janela",
               cont_invalidas == 0,
               f"{cont_invalidas} janelas incompletas"))

conjunto_cont = ranking_cont_lido.filter(F.col("indicador") == "dec_fi").select("num_cnpj")
fora_conjunto = (conjunto_qualidade.join(conjunto_cont, "num_cnpj", "left_anti").count() +
                 conjunto_cont.join(conjunto_qualidade, "num_cnpj", "left_anti").count())
testes.append(("continuidade no conjunto de Qualidade",
               fora_conjunto == 0,
               f"{fora_conjunto} distribuidoras fora do conjunto comum"))

# The window ending in December 2025 is calendar year 2025 in the continuity Gold
anual_existe = spark.sql(f"SHOW TABLES IN {GOLD} LIKE 'fato_continuidade_anual'").count() > 0
if anual_existe:
    anual = (spark.table(f"{GOLD}.fato_continuidade_anual").filter(F.col("ano") == 2025)
        .select("num_cnpj", *[F.col(n).alias(f"a_{n}") for n in INDICADORES_CONT]))
    comparados = cont_lida.filter(F.col("fim_janela") == 202512).join(anual, "num_cnpj")
    desvios = comparados.filter(" OR ".join(
        f"abs({n} - a_{n}) > 0.01" for n in INDICADORES_CONT)).count()
    testes.append(("janela dez/2025 reproduz a Gold anual",
                   desvios == 0 and comparados.count() > 0,
                   f"{desvios} de {comparados.count()} distribuidoras com desvio acima de 0,01"))
else:
    testes.append(("janela dez/2025 reproduz a Gold anual",
                   False,
                   "gold.fato_continuidade_anual nao encontrada"))

for nome, passou, detalhe in testes:
    print(f"[{'OK' if passou else 'FALHOU':<7}] {nome:<45} {detalhe}")

if all(p for _, p, _ in testes):
    print("\nGold de reclamacoes validada.")
else:
    print("\nHa teste sem passar; corrigir antes de seguir para a analise.")

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Discussão dos resultados | Rankings e janelas prontos | Análise e narrativa no `04_analysis` |
| Inferência estatística | Variações descritivas | Registrada como trabalho futuro na autoavaliação final |

## Autoavaliação desta etapa

### O que a etapa entregou

Quatro tabelas na Gold: `fato_reclamacao_janela`, com cinco recortes em quatro janelas móveis de 12 meses para as 33 distribuidoras de grande porte; `ranking_reclamacoes`, com dez rankings (cinco recortes por procedentes e por recebidas) e a sensibilidade sem imputação; `continuidade_janela` e `ranking_continuidade`, com o DEC-FI e o FEC-FI recalculados nas mesmas janelas. A janela encerrada em dezembro de 2025 reproduz a Gold anual de continuidade nas 33 distribuidoras, o que valida a aplicação das equações do PRODIST a janelas móveis.

### O que mudou no caminho

- O desenho inicial previa dois rankings: o comercial estrito e o item 285. O Pareto de 2024 mostrou que toda soma é dominada por um grupo (Qualidade responde por 93,7% das procedentes da família 102; Faturamento por 51,3% do comercial estrito), e os rankings passaram a ser por grupo, com o total mantido como resposta direta à pergunta. O item 285 e Rede/Manutenção saíram.
- O ranking de recebidas entrou em paralelo ao de procedentes, porque a procedência é classificada pela própria distribuidora.
- A justificativa de usar só o nível 1 mudou da escala para a dupla contagem: no comercial estrito, o nível 2 equivale a cerca de um quinto do nível 1, e o argumento de escala não se sustentava.
- A continuidade foi recalculada nas mesmas janelas das reclamações, porque a Gold de continuidade compara anos civis e não chega a 2026.
- O nível 2 entrou na tabela de janela, restrito ao total do comercial estrito, depois que a comparação por tipologia mostrou inconsistência entre os níveis.

### O que ficou em aberto

**Inconsistência de tipologia entre os níveis 1 e 2.** Em ENEL CE, EMS, EMR e ESE há tipologias em que o nível 2 recebe mais reclamações do que o nível 1 inteiro, todos os meses da série (na ENEL CE, `1020701` tem zero no nível 1 e entre 59 e 166 por mês no nível 2). A causa não foi apurada. Levantei três hipóteses:

1. Inversão de nível no reporte, sistemática nessas tipologias.
2. Codificação distinta entre os níveis: a reclamação entra no nível 1 com um código e chega à ouvidoria com outro. Considero pouco provável que isso, sozinho, produza volume capaz de deformar 12 meses.
3. Uso distinto de tipologias de baixa incidência. Todas as tipologias afetadas têm volume muito baixo no nível 1, o que sugere que o atendimento de primeiro nível praticamente não as usa e que elas se ajustam melhor ao tipo de caso que chega à ouvidoria. Como muitas tipologias da REH 2.992/2021 não têm critério claro de uso, a confusão tende a se concentrar nas de menor incidência.

A consequência para o trabalho é restringir a comparação entre níveis ao total do comercial estrito, onde o nível 2 é menor que o nível 1 em todas as distribuidoras.

**Efeito de eventos climáticos na base de comparação.** A janela inicial (dezembro de 2024) contém as enchentes do Rio Grande do Sul em maio de 2024, e a ELETROPAULO sofreu eventos climáticos severos nos novembros dos últimos três anos. Parte da variação do DEC-FI dessas empresas é efeito de base, e não de gestão. Registrei como melhoria um estudo em notebook próprio, com a série mensal dessas distribuidoras e um ranking paralelo que neutraliza os meses de evento, a ser feito somente depois de o trabalho estar concluído.

**Último mês da série.** A Neoenergia Brasília em junho de 2026 ficou sem imputação, por não haver meses seguintes que distingam anomalia de novo patamar. Com julho de 2026 consolidado, a decisão pode ser revista.

### O que eu faria diferente

Eu teria olhado a composição por grupo antes de desenhar os rankings. O desenho inicial, com dois recortes somados, foi feito sobre a regra normativa, e só o Pareto mostrou que a soma escondia a leitura por grupo. Também teria comparado os níveis por tipologia antes de propor a taxa de escalada: a inconsistência entre níveis mudou a forma de medir a escalada e só apareceu quando os valores absolutos foram abertos.